In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Define your CSV files here
csv_files = [
    "work_dirs/fedavg_iid/roc_prc.csv",
    "work_dirs/fedavg_iid_gn/roc_prc.csv",
    "work_dirs/fedavg_dirichlet/roc_prc.csv",
    "work_dirs/fedavg_dirichlet_gn/roc_prc.csv",
    "work_dirs/fedavg_kmeans/roc_prc.csv",
    "work_dirs/fedavg_kmeans_groupnorm/roc_prc.csv",
    "work_dirs/fedavg_feature_hierarchical/roc_prc.csv",
    "work_dirs/fedavg_feature_hierarchical_groupnorm/roc_prc.csv",
]

result = []
roc_curves = []  # Store ROC data for plotting

for csv_file in csv_files:
    print(f"Processing {csv_file}...")
    
    df = pd.read_csv(csv_file, header=None, names=["threshold", "id", "tn", "fp", "fn", "tp"])
    
    t = df
    no_negatives      = (t["fp"] == 0) & (t["tn"] == 0)
    no_positives      = (t["tp"] == 0) & (t["fn"] == 0)
    no_pred_positives = (t["tp"] == 0) & (t["fp"] == 0)
    df = t[~(no_negatives | no_positives | no_pred_positives)].copy() 
    
    df_filtered = df[df["threshold"] == 0.1].copy()
    
    m_result = {}
    m_result['tp_0.10'] = df_filtered['tp'].sum()
    m_result['fp_0.10'] = df_filtered['fp'].sum()
    m_result['tn_0.10'] = df_filtered['tn'].sum()
    m_result['fn_0.10'] = df_filtered['fn'].sum()
    m_result['tpr_10%'] = m_result['tp_0.10'] / (m_result['tp_0.10'] + m_result['fn_0.10'])
    m_result['accuracy at 10% threshold'] = (m_result['tp_0.10'] + m_result['tn_0.10']) / (m_result['tp_0.10'] + m_result['tn_0.10'] + m_result['fp_0.10'] + m_result['fn_0.10'])
    m_result['precision at 10% threshold'] = m_result['tp_0.10'] / (m_result['tp_0.10'] + m_result['fp_0.10'])
    m_result['recall at 10% threshold'] = m_result['tp_0.10'] / (m_result['tp_0.10'] + m_result['fn_0.10'])
    m_result['f1 at 10% threshold'] = 2 * m_result['precision at 10% threshold'] * m_result['recall at 10% threshold'] / (m_result['precision at 10% threshold'] + m_result['recall at 10% threshold'])
    
    m_result['tp'] = df['tp'].sum()
    m_result['fp'] = df['fp'].sum()
    m_result['tn'] = df['tn'].sum()
    m_result['fn'] = df['fn'].sum()
    m_result['accuracy'] = (m_result['tp'] + m_result['tn']) / (m_result['tp'] + m_result['tn'] + m_result['fp'] + m_result['fn'])
    
    # ---------- ROC curve ----------
    df.loc[:, "pos"] = df["tp"] + df["fn"]
    df.loc[:, "neg"] = df["fp"] + df["tn"]
    
    valid = df[(df["pos"] > 0) & (df["neg"] > 0)].copy()
    valid["tpr"] = valid["tp"] / valid["pos"]
    valid["fpr"] = valid["fp"] / valid["neg"]
    valid["tnr"] = valid["tn"] / valid["neg"]
    macro = valid.groupby("threshold")[["fpr", "tpr"]].mean()
    
    def roc_points_and_auc(roc):
        r = roc.sort_index(ascending=False)
        fpr = np.r_[0.0, r["fpr"].to_numpy(), 1.0]
        tpr = np.r_[0.0, r["tpr"].to_numpy(), 1.0]
        auc = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)
        return fpr, tpr, auc
    
    fpr_ma, tpr_ma, auc_ma = roc_points_and_auc(macro)
    m_result['auc'] = auc_ma
    
    # Extract tag from file path
    tag = Path(csv_file).parent.name
    m_result['tag'] = tag
    
    result.append(m_result)
    roc_curves.append({
        'tag': tag,
        'fpr': fpr_ma,
        'tpr': tpr_ma,
        'auc': auc_ma
    })

# ---------- Combined ROC plot ----------
FPR_MAX = 0.10

def partial_auc(fpr, tpr, fpr_max):
    tpr_cut = np.interp(fpr_max, fpr, tpr)
    mask = fpr <= fpr_max
    f = np.r_[fpr[mask], fpr_max]
    t = np.r_[tpr[mask], tpr_cut]
    return np.sum(np.diff(f) * (t[1:] + t[:-1]) / 2)

fig, ax = plt.subplots(figsize=(8, 8))

for curve in roc_curves:
    pauc = partial_auc(curve['fpr'], curve['tpr'], FPR_MAX)
    ax.plot(curve['fpr'], curve['tpr'], marker=".", markersize=3,
            label=f"{curve['tag']}, AUC = {curve['auc']:.4f}")

ax.set_xlabel("False positive rate", fontsize=11)
ax.set_ylabel("True positive rate", fontsize=11)
ax.set_title(f"ROC Comparison (FPR ≤ {FPR_MAX:.0%})", fontsize=12)
ax.set_xlim(0, FPR_MAX)
ax.set_ylim(0, 1)
ax.set_box_aspect(1)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

# Display results table
display(pd.DataFrame(result, columns=[
    "tag",
    "tp_0.10", "fp_0.10", "tn_0.10", "fn_0.10",
    "accuracy at 10% threshold", "precision at 10% threshold", "recall at 10% threshold", "f1 at 10% threshold",
    "tp", "fp", "tn", "fn", "accuracy", "auc"
]))


Processing work_dirs/fedavg_iid/roc_prc.csv...
Processing work_dirs/fedavg_iid_gn/roc_prc.csv...
Processing work_dirs/fedavg_dirichlet/roc_prc.csv...
Processing work_dirs/fedavg_dirichlet_gn/roc_prc.csv...
Processing work_dirs/fedavg_kmeans/roc_prc.csv...
Processing work_dirs/fedavg_kmeans_groupnorm/roc_prc.csv...
Processing work_dirs/fedavg_feature_hierarchical/roc_prc.csv...
Processing work_dirs/fedavg_feature_hierarchical_groupnorm/roc_prc.csv...


<Figure size 800x800 with 1 Axes>

,tag,tp_0.10,fp_0.10,tn_0.10,fn_0.10,accuracy at 10% threshold,precision at 10% threshold,recall at 10% threshold,f1 at 10% threshold,tp,fp,tn,fn,accuracy,auc
0,fedavg_iid,1303474,994790,80828133,825219,0.978321,0.567156,0.612335,0.588880,90291972,223561554,7241023522,195541384,0.945925,0.919624
1,fedavg_iid_gn,1408630,957368,80733750,720796,0.979979,0.595364,0.661507,0.626695,103372200,182532416,7913242904,186183680,0.956028,0.944461
2,fedavg_dirichlet,1185098,1017415,79627762,941693,0.976331,0.538066,0.557224,0.547477,82734550,216283529,6945319541,193080876,0.944959,0.910477
3,fedavg_dirichlet_gn,1307894,794391,81552092,821527,0.980871,0.622130,0.614202,0.618140,92634378,173552077,7034544221,176271564,0.953213,0.946410
4,fedavg_kmeans,874786,325022,80777388,1253524,0.981034,0.729105,0.411024,0.525694,54352208,170697351,6310193879,178657618,0.947965,0.940496
5,fedavg_kmeans_groupnorm,1296415,850679,81102579,833015,0.979976,0.603800,0.608808,0.606294,88325546,172971922,6964357749,173293647,0.953201,0.951204
6,fedavg_feature_hierarchical,952324,378727,78954630,1175567,0.980920,0.715468,0.447544,0.550645,63927784,167895069,6190593707,182104912,0.947006,0.935738
7,fedavg_feature_hierarchical_groupnorm,1318615,1017905,81459611,810845,0.978385,0.564350,0.619225,0.590515,93184656,194292581,7736028423,176358660,0.954798,0.950552
